In [ ]:
import os, sys, platform
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import torch, numpy as np, pandas as pd
from pathlib import Path

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "Transformer":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Python : {sys.version.split()[0]}")
print(f"torch : {torch.__version__}")
print(f"System : {platform.platform()}")
print(f"CUDA : {torch.cuda.is_available()}")
print(f"MPS : {torch.backends.mps.is_available()}")
print(f"Repo : {REPO_ROOT}")


Python : 3.11.11
torch  : 2.11.0
System : macOS-26.4.1-arm64-arm-64bit
CUDA   : False
MPS    : True
Repo   : /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main


In [2]:
import shutil

CLEAN_DIRS = [
    REPO_ROOT / "Transformer" / d for d in
    ["checkpoints", "split_labels", "val_labels",
     "derived_labels", "outputs", "figures"]
]
for d in CLEAN_DIRS:
    if d.exists() and any(d.iterdir()):
        n = sum(1 for _ in d.rglob("*") if _.is_file())
        shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
        print(f"Cleared {d.relative_to(REPO_ROOT)} ({n} items)")
    else:
        d.mkdir(parents=True, exist_ok=True)
        print(f"{d.relative_to(REPO_ROOT)}: already empty")
print("\nAll output directories clean. Ready for a fresh run.")


Transformer/checkpoints: already empty
Transformer/split_labels: already empty
Transformer/val_labels: already empty
Transformer/derived_labels: already empty
Transformer/outputs: already empty
Transformer/figures: already empty

All output directories clean. Ready for a fresh run.


In [ ]:
from Transformer.config.model_config import ModelConfig, DEVICE

config = ModelConfig()
config.validate()
torch.manual_seed(config.random_seed)
np.random.seed(config.random_seed)

print(f"Device = {DEVICE}")
print(f"n_risks = {config.n_risks}")
print(f"t_grid = {config.t_grid}")
print(f"batch_size = {config.batch_size_physical} (eff={config.batch_size_physical * config.gradient_accumulation_steps})")
print(f"n_grid = {config.n_grid}")
print(f"v_max = {config.v_max}")
print(f"d_model = {config.d_model}")


Device     = mps
n_risks    = 1
t_grid     = [12, 24, 36, 48, 60]
batch_size = 2 (eff=16)
n_grid     = 5
v_max      = 10
d_model    = 512


In [ ]:
MASTER_CSV = REPO_ROOT / "data" / "master_data_improved_04052026_v3.csv"
DVF_DIR    = REPO_ROOT / "data" / "dvf_structured"

assert DVF_DIR.is_dir(), f"DVF directory not found: {DVF_DIR}"
dvf_subjects = sorted([d.name for d in DVF_DIR.iterdir() if d.is_dir()])
dvf_set = set(dvf_subjects)
print(f"DVF folders : {len(dvf_subjects)}")

df = pd.read_csv(str(MASTER_CSV), low_memory=False)
print(f"Master CSV : {len(df):,} rows, {df['PTID'].nunique()} patients")

df_bl = df[df["VISCODE"] == "bl"].drop_duplicates("PTID")
mci_bl = df_bl[df_bl["DX_bl"].isin(["LMCI", "EMCI"])]
print(f"MCI-at-baseline: {len(mci_bl)}")

from Transformer.train import build_labels_from_csv
subject_ids, durations, events = build_labels_from_csv(MASTER_CSV, dvf_set)

n_events = int((events == 1).sum())
n_censor = int((events == 0).sum())
print(f"Labeled MCI with DVF: {len(subject_ids)} (events={n_events}, censored={n_censor})")


DVF folders : 1235
Master CSV  : 16,421 rows, 2430 patients
MCI-at-baseline: 1113


mamba_ssm package not found — using sequential PyTorch fallback for selective scan. For production use on CUDA, install via: pip install mamba-ssm causal-conv1d


Labeled MCI with DVF: 522 (events=230, censored=292)


In [ ]:
from sklearn.model_selection import train_test_split
from Transformer.data.label_io import save_survival_labels

SPLIT_DIR = REPO_ROOT / "Transformer" / "split_labels"
VAL_DIR   = REPO_ROOT / "Transformer" / "val_labels"

idx = np.arange(len(subject_ids))
train_idx, val_idx = train_test_split(
    idx, test_size=0.2, random_state=config.random_seed, stratify=events,
)

train_ids = [subject_ids[i] for i in train_idx]
val_ids   = [subject_ids[i] for i in val_idx]
train_dur, val_dur = durations[train_idx], durations[val_idx]
train_evt, val_evt = events[train_idx], events[val_idx]

save_survival_labels(SPLIT_DIR, train_ids, train_dur, train_evt)
save_survival_labels(VAL_DIR,   val_ids,   val_dur,   val_evt)

tr_rate = train_evt.mean()
va_rate = val_evt.mean()
print(f"Train: {len(train_ids)} subjects, {int(train_evt.sum())} events ({tr_rate:.1%})")
print(f"Val  : {len(val_ids)} subjects, {int(val_evt.sum())} events ({va_rate:.1%})")
assert abs(tr_rate - va_rate) < 0.02, f"Event rate drift too large: {abs(tr_rate - va_rate):.3f}"
print(f"Event rate delta = {abs(tr_rate - va_rate):.4f} pass ")


Train: 417 subjects, 184 events (44.1%)
Val  : 105 subjects, 46 events (43.8%)
Event rate delta = 0.0032 ✓


In [ ]:
from Transformer.data.dvf_dataset import NormalizationStats

train_dvf_paths = []
for sid in train_ids:
    train_dvf_paths.extend(sorted((DVF_DIR / sid).glob("*.npy")))
print(f"Training DVF files: {len(train_dvf_paths)}")

norm_stats = NormalizationStats.compute(train_dvf_paths)

for c, axis in enumerate(["delta x", "delta y", "delta z"]):
    print(f"  {axis}: μ={norm_stats.mean[c]:.6f}, σ={norm_stats.std[c]:.6f}")
print("Pass Normalization stats computed")


Training DVF files: 1739
  Δx: μ=0.001725, σ=0.567706
  Δy: μ=0.005662, σ=0.736165
  Δz: μ=0.001994, σ=1.494016
✓ Normalization stats computed


In [7]:
import multiprocessing as mp
from Transformer.data.dvf_dataset import LongitudinalDVFDataset

train_ds = LongitudinalDVFDataset(
    subject_ids=train_ids, dvf_dir=DVF_DIR, config=config,
    norm_stats=norm_stats, survival_labels_dir=SPLIT_DIR,
)
val_ds = LongitudinalDVFDataset(
    subject_ids=val_ids, dvf_dir=DVF_DIR, config=config,
    norm_stats=norm_stats, survival_labels_dir=VAL_DIR,
)

NW = 2
ctx = mp.get_context("spawn")
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=config.batch_size_physical, shuffle=True,
    num_workers=NW, persistent_workers=True,
    multiprocessing_context=ctx, pin_memory=False,
    collate_fn=train_ds.collate_fn,
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=config.batch_size_physical, shuffle=False,
    num_workers=NW, persistent_workers=True,
    multiprocessing_context=ctx, pin_memory=False,
    collate_fn=val_ds.collate_fn,
)

print(f"Train: {len(train_ds)} subjects, {len(train_loader)} batches")
print(f"Val  : {len(val_ds)} subjects, {len(val_loader)} batches")
s0 = train_ds[0]
print(f"Sample 0: {s0['subject_id']}, DVF={tuple(s0['dvf_sequence'].shape)}, "
      f"dur={s0['duration']:.1f}mo, event={s0['event']}")


Train: 417 subjects, 209 batches
Val  : 105 subjects, 53 batches
Sample 0: 099_S_0958, DVF=(10, 3, 128, 128, 128), dur=6.0mo, event=0


In [ ]:
from Transformer.losses.ipcw_loss import CensoringSurvivalEstimator
from Transformer.utils.chi_interpolation import ConstantHazardInterpolator

censor_est = CensoringSurvivalEstimator()
censor_est.fit(train_ds.durations, train_ds.events)

print("G(t) at grid points:")
for t in config.t_grid:
    g = censor_est(np.array([t]))[0]
    print(f"  G({t:3d} mo) = {g:.4f}")

chi = ConstantHazardInterpolator.from_config(config)
print(f"Pass CHI interpolator: grid={config.t_grid}")


G(t) at grid points:
  G( 12 mo) = 0.8962
  G( 24 mo) = 0.8166
  G( 36 mo) = 0.6175
  G( 48 mo) = 0.4900
  G( 60 mo) = 0.3960

✓ CHI interpolator: grid=[12, 24, 36, 48, 60]


In [ ]:
from Transformer.models.pipeline import ADNISurvivalPipeline

model = ADNISurvivalPipeline(config=config)

total  = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - train_
print(f"Total params: {total:,}")
print(f"Trainable: {train_:,}")
print(f"Frozen: {frozen:,}")

components = {
    "extractor": model.extractor,
    "longformer": model.longformer,
    "mamba": model.mamba,
    "gated_fusion": model.gated_fusion,
    "pma": model.pma,
    "survival_head": model.survival_head,
}
for name, mod in components.items():
    if mod is None:
        continue
    n = sum(p.numel() for p in mod.parameters())
    t = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    print(f"  {name:20s}: {n:>10,} params ({t:,} trainable)")

print(f"\nDevice: {DEVICE}")


Total params    : 73,878,981
Trainable       : 27,680,005
Frozen          : 46,198,976
  extractor           : 48,825,024 params (2,626,048 trainable)
  longformer          : 19,179,008 params (19,179,008 trainable)
  mamba               :  2,677,760 params (2,677,760 trainable)
  gated_fusion        :    524,800 params (524,800 trainable)
  pma                 :  1,582,080 params (1,582,080 trainable)
  survival_head       :  1,090,309 params (1,090,309 trainable)

Device: mps


In [ ]:
from Transformer.training.trainer import LPFTTrainer

trainer = LPFTTrainer(
    model=model, config=config,
    train_loader=train_loader, val_loader=val_loader,
    censoring_estimator=censor_est, chi_interpolator=chi,
    train_durations=train_ds.durations,
    train_events=train_ds.events,
    norm_stats=norm_stats,
)

print(f"Device: {trainer.device}")
print(f"AMP enabled: {trainer._amp_enabled}")
print(f"AMP dtype: {trainer._amp_dtype}")
print(f"Grad accum: {config.gradient_accumulation_steps}")
print(f"Grad clip: {config.grad_clip_max_norm}")
print(f"Scaler type: {type(trainer.scaler).__name__}")
print("Pass Trainer ready")


Device         : mps
AMP enabled    : True
AMP dtype      : torch.float16
Grad accum     : 8
Grad clip      : 1.0
Scaler type    : GradScaler
✓ Trainer ready


In [ ]:
print("STAGE 1 — Linear Probe")
stage1_metrics = trainer.run_stage1()
print(f"\nStage 1 done: C_td={stage1_metrics['c_td']:.4f}, "
      f"Uno_C={stage1_metrics['uno_c']:.4f}, IBS={stage1_metrics['ibs']:.4f}")


STAGE 1 — Linear Probe


/opt/anaconda3/envs/torch_bedrock/lib/python3.11/site-packages/SurvivalEVAL/Evaluator.py:71: UserWarning: The first time coordinate is not 0. A authentic survival curve should start from 0 with 100% survival probability. Adding 0 to the beginning of the time coordinates and 1 to the beginning of the predicted curves.
  warnings.warn(zero_pad_msg)
/opt/anaconda3/envs/torch_bedrock/lib/python3.11/site-packages/SurvivalEVAL/Evaluator.py:71: UserWarning: The first time coordinate is not 0. A authentic survival curve should start from 0 with 100% survival probability. Adding 0 to the beginning of the time coordinates and 1 to the beginning of the predicted curves.
  warnings.warn(zero_pad_msg)
/opt/anaconda3/envs/torch_bedrock/lib/python3.11/site-packages/SurvivalEVAL/Evaluator.py:71: UserWarning: The first time coordinate is not 0. A authentic survival curve should start from 0 with 100% survival probability. Adding 0 to the beginning of the time coordinates and 1 to the beginning of the p


Stage 1 done: C_td=0.4465, Uno_C=0.4516, IBS=0.2633


In [ ]:
print("STAGE 2 — Full Fine-Tune")
best = trainer.run_stage2()
print(f"\nBest C_td = {best.get('c_td', 0):.4f} @ epoch {trainer.best_epoch}")


STAGE 2 — Full Fine-Tune


/opt/anaconda3/envs/torch_bedrock/lib/python3.11/site-packages/SurvivalEVAL/Evaluator.py:71: UserWarning: The first time coordinate is not 0. A authentic survival curve should start from 0 with 100% survival probability. Adding 0 to the beginning of the time coordinates and 1 to the beginning of the predicted curves.
  warnings.warn(zero_pad_msg)
/opt/anaconda3/envs/torch_bedrock/lib/python3.11/site-packages/SurvivalEVAL/Evaluator.py:71: UserWarning: The first time coordinate is not 0. A authentic survival curve should start from 0 with 100% survival probability. Adding 0 to the beginning of the time coordinates and 1 to the beginning of the predicted curves.
  warnings.warn(zero_pad_msg)
/opt/anaconda3/envs/torch_bedrock/lib/python3.11/site-packages/SurvivalEVAL/Evaluator.py:71: UserWarning: The first time coordinate is not 0. A authentic survival curve should start from 0 with 100% survival probability. Adding 0 to the beginning of the time coordinates and 1 to the beginning of the p

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from Transformer.metrics.concordance import compute_all_metrics, format_comparison_table

# Load best checkpoint
ckpt_path = config.checkpoint_dir / "checkpoint_best.pt"
if ckpt_path.exists():
    trainer.load_checkpoint(ckpt_path)
    print(f"Loaded {ckpt_path}")

# Final validation
model.eval()
all_hazards, all_dur, all_evt = [], [], []
with torch.no_grad():
    for batch in val_loader:
        dvf = batch["dvf_sequence"].to(DEVICE)
        td  = batch["time_deltas"].to(DEVICE)
        mm  = batch["missing_mask"].to(DEVICE)
        out = model(dvf, td, mm)
        all_hazards.append(out["hazards"].float().cpu())
        all_dur.append(batch["duration"].numpy())
        all_evt.append(batch["event"].numpy())

hazards_cat = torch.cat(all_hazards)
dur_cat = np.concatenate(all_dur)
evt_cat = np.concatenate(all_evt)

final_metrics = compute_all_metrics(
    hazards_cat, dur_cat, evt_cat,
    train_ds.durations, train_ds.events,
    chi, model.survival_head, config,
)

print(f"Antolini C_td: {final_metrics['c_td']:.4f}║")
print(f"Uno C_τ (τ={final_metrics['tau']:.0f}mo):  {final_metrics['uno_c']:.4f}║")
print(f"IBS: {final_metrics['ibs']:.4f}║")

# Figures
fig_dir = config.figures_dir
fig_dir.mkdir(parents=True, exist_ok=True)

# KM overall + risk tertiles
from Transformer.models.survival_head import TraCeRSurvivalHead
S_last = model.survival_head.hazards_to_survival(hazards_cat).numpy()[:, -1]
risk = -S_last
tertiles = np.percentile(risk, [33.3, 66.7])
groups = np.digitize(risk, tertiles)

from lifelines import KaplanMeierFitter
fig, ax = plt.subplots(figsize=(8, 5))
for g, label in enumerate(["Low risk", "Medium risk", "High risk"]):
    mask = groups == g
    kmf = KaplanMeierFitter()
    kmf.fit(dur_cat[mask], evt_cat[mask], label=label)
    kmf.plot_survival_function(ax=ax)
ax.set_xlabel("Months"); ax.set_ylabel("Survival probability")
ax.set_title("KM Curves by Predicted Risk Tertile")
fig.savefig(fig_dir / "km_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved {fig_dir / 'km_curves.png'}")

# CIF at grid points
S_all = model.survival_head.hazards_to_survival(hazards_cat).numpy()
fig, ax = plt.subplots(figsize=(8, 5))
mean_s = S_all.mean(axis=0)
ax.step(config.t_grid, 1 - mean_s, where="mid", linewidth=2, color="#4e79a7")
ax.fill_between(config.t_grid, 0, 1 - mean_s, alpha=0.15, step="mid", color="#4e79a7")
ax.set_xlabel("Months"); ax.set_ylabel("Cumulative incidence")
ax.set_title("Mean Cumulative Incidence Function")
fig.savefig(fig_dir / "cif_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved {fig_dir / 'cif_curves.png'}")

# Calibration plot
fig, ax = plt.subplots(figsize=(6, 6))
pred_risk = risk
deciles = pd.qcut(pred_risk, 10, duplicates="drop")
cal_df = pd.DataFrame({"pred": pred_risk, "event": evt_cat, "decile": deciles})
grouped = cal_df.groupby("decile", observed=True).agg(
    pred_mean=("pred", "mean"), obs_rate=("event", "mean")
)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
ax.scatter(grouped["pred_mean"], grouped["obs_rate"], s=60, zorder=5)
ax.set_xlabel("Predicted risk (mean per decile)")
ax.set_ylabel("Observed event rate")
ax.set_title("Calibration Plot")
ax.legend()
fig.savefig(fig_dir / "calibration.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved {fig_dir / 'calibration.png'}")


In [ ]:
table_str, _ = format_comparison_table(final_metrics)
print(table_str)

out_dir = REPO_ROOT / "Transformer" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

baseline_csv = REPO_ROOT / "Baseline" / "outputs" / "model_comparison.csv"
if baseline_csv.exists():
    bl = pd.read_csv(str(baseline_csv))
    new_row = pd.DataFrame([{
        "Model": "Transformer (Parallel LF+Mamba)",
        "Cohort": "MCI → Dementia",
        "Metric": "Antolini C_td",
        "Value": final_metrics["c_td"],
    }])
    combined = pd.concat([bl, new_row], ignore_index=True)
    combined.to_csv(str(out_dir / "model_comparison_with_transformer.csv"), index=False)
    print(f"\n✓ Saved {out_dir / 'model_comparison_with_transformer.csv'}")
else:
    print(f"Baseline CSV not found at {baseline_csv} — skipping merge.")


In [ ]:
print(f"Best C_td: {trainer.best_ctd:.4f}")
print(f"Best epoch: {trainer.best_epoch}")
print(f"Checkpoint: {config.checkpoint_dir / 'checkpoint_best.pt'}")
print(f"Figures: {config.figures_dir}")
print("Pipeline complete Pass")
